In [2]:
#importar bibliotecas
import pyspark
from pyspark import SparkFiles

#criar spark session
sc = pyspark.SparkContext('local')

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=pyspark-shell, master=local) created by __init__ at /tmp/ipykernel_1697/2391376810.py:6 

In [3]:
#creating a PairRDD from a list

list_tuples = [(1,"Banana"), (2, "Melão"), (3, "Melancia")]

pair_rdd = sc.parallelize(list_tuples)
pair_rdd.collect()

[(1, 'Banana'), (2, 'Melão'), (3, 'Melancia')]

In [7]:
#Criar a partir de um RDD
rdd = sc.parallelize(["12345, Abacate","12346, Banana", "12347, Maça"])
pair_rdd = rdd.map(lambda x: (x.split(",")[0],x.split(",")[1]))

pair_rdd.take(5)

[('12345', ' Abacate'), ('12346', ' Banana'), ('12347', ' Maça')]

In [9]:
#criar função extra
def criar_rdd(rdd):
  y = rdd.split(",")
  return (y[0],y[1], y[0])

rdd = sc.parallelize(["12345, Abacate","12346, Banana", "12347, Maça"])
pair_rdd = rdd.map(criar_rdd)

pair_rdd.take(5)

[('12345', ' Abacate', '12345'),
 ('12346', ' Banana', '12346'),
 ('12347', ' Maça', '12347')]

In [11]:
#Exercício 1

#copiar url
url = 'https://raw.githubusercontent.com/joegotz/bigDataAnalytics/main/euro2024_players.csv'

#adicionar arquivo ao contexto
sc.addFile(url)

In [ ]:
#Exercício 1
#1. criar rdd a partir do arquivo baixado
#2. Transformar o rdd em pairRdd -- (name, country)
#3. Aplicar filter para deixar apenas dados da Alemanha
#4. Capitalize only the values of Country
#5. Realizar ordenação baseado na chave (nome do jogador)

In [ ]:
#Exercício 1
#1. criar rdd a partir do arquivo baixado
rdd = sc.textFile(SparkFiles.get('euro2024_players.csv'))
rdd.take(5)

['Name,Position,Age,Club,Height,Foot,Caps,Goals,MarketValue,Country',
 'Marc-André ter Stegen,Goalkeeper,32,FC Barcelona,187,right,40,0,28000000,Germany',
 'Manuel Neuer,Goalkeeper,38,Bayern Munich,193,right,119,0,4000000,Germany',
 'Oliver Baumann,Goalkeeper,34,TSG 1899 Hoffenheim,187,right,0,0,3000000,Germany',
 'Nico Schlotterbeck,Centre-Back,24,Borussia Dortmund,191,left,12,0,40000000,Germany']

In [ ]:
#2. Transformar o rdd em pairRdd -- (name, country)
pairRdd = rdd.\
  map(lambda x: (x.split(",")[0], x.split(",")[-1]))

pairRdd.take(5)

[('Name', 'Country'),
 ('Marc-André ter Stegen', 'Germany'),
 ('Manuel Neuer', 'Germany'),
 ('Oliver Baumann', 'Germany'),
 ('Nico Schlotterbeck', 'Germany')]

In [ ]:
#3. Aplicar filter para deixar apenas dados da Alemanha
pairRdd = pairRdd.filter(lambda x: x[1]=='Germany')
pairRdd.take(5)

[('Marc-André ter Stegen', 'Germany'),
 ('Manuel Neuer', 'Germany'),
 ('Oliver Baumann', 'Germany'),
 ('Nico Schlotterbeck', 'Germany'),
 ('Jonathan Tah', 'Germany')]

In [ ]:
#4. Capitalize only the values of Country
pairRdd = pairRdd.mapValues(lambda x: x.lower())
pairRdd.take(5)

[('Marc-André ter Stegen', 'germany'),
 ('Manuel Neuer', 'germany'),
 ('Oliver Baumann', 'germany'),
 ('Nico Schlotterbeck', 'germany'),
 ('Jonathan Tah', 'germany')]

In [ ]:
#5. Realizar ordenação baseado na chave (nome do jogador)
pairRdd = pairRdd.sortByKey(ascending=False)
pairRdd.take(5)

[('İlkay Gündoğan', 'germany'),
 ('Waldemar Anton', 'germany'),
 ('Toni Kroos', 'germany'),
 ('Thomas Müller', 'germany'),
 ('Robin Koch', 'germany')]

In [ ]:
#### Challenge
#1. Load the Euro2024.csv dataset as a conventional RDD and remove the header
#2. Convert it into a PairRDD with (Name, (Age, Goals,Country)) format
#3. Obtain all entries in which the Country is equal to GERMANY and AGE <25
#4. Capitalize only the Country, but maintaining the format (k, (v1,v2,v3))
#5. Sort the results according to the Age
#6. Average age for country using reduceByKey
#7. Average goals for country reduceByKey

### Solução do desafio

In [12]:
# 1. Load the Euro2024.csv dataset as a conventional RDD and remove the header
rdd_challenge = sc.textFile(SparkFiles.get('euro2024_players.csv'))

header = rdd_challenge.first() # pegou o header
data_rdd = rdd_challenge.filter(lambda row: row != header) # tirou o header
data_rdd.take(5)

['Marc-André ter Stegen,Goalkeeper,32,FC Barcelona,187,right,40,0,28000000,Germany',
 'Manuel Neuer,Goalkeeper,38,Bayern Munich,193,right,119,0,4000000,Germany',
 'Oliver Baumann,Goalkeeper,34,TSG 1899 Hoffenheim,187,right,0,0,3000000,Germany',
 'Nico Schlotterbeck,Centre-Back,24,Borussia Dortmund,191,left,12,0,40000000,Germany',
 'Jonathan Tah,Centre-Back,28,Bayer 04 Leverkusen,195,right,25,0,30000000,Germany']

In [15]:
# 2. Convert it into a PairRDD with (Name, (Age, Goals,Country)) format
def parse_player_data(line):
    fields = line.split(',')
    name = fields[0]
    age = int(fields[2])
    goals = int(fields[7])
    country = fields[9]
    return (name, (age, goals, country))

pair_rdd_challenge = data_rdd.map(parse_player_data)

pair_rdd_challenge.take(5)

[('Marc-André ter Stegen', (32, 0, 'Germany')),
 ('Manuel Neuer', (38, 0, 'Germany')),
 ('Oliver Baumann', (34, 0, 'Germany')),
 ('Nico Schlotterbeck', (24, 0, 'Germany')),
 ('Jonathan Tah', (28, 0, 'Germany'))]

In [17]:
# 3. Obtain all entries in which the Country is equal to GERMANY and AGE <25
filtered_rdd = pair_rdd_challenge.filter(lambda x: x[1][2].upper() == 'GERMANY' and x[1][0] < 25)

filtered_rdd.take(5)

[('Nico Schlotterbeck', (24, 0, 'Germany')),
 ('Aleksandar Pavlovic', (20, 0, 'Germany')),
 ('Florian Wirtz', (21, 1, 'Germany')),
 ('Jamal Musiala', (21, 2, 'Germany')),
 ('Kai Havertz', (24, 16, 'Germany'))]

In [18]:
# 4. Capitalize only the Country, but maintaining the format (k, (v1,v2,v3))
def capitalize_country_tuple(x):
    name = x[0]
    age = x[1][0]
    goals = x[1][1]
    country = x[1][2].upper()
    return (name, (age, goals, country))

capitalized_rdd = filtered_rdd.map(capitalize_country_tuple)

capitalized_rdd.take(5)

[('Nico Schlotterbeck', (24, 0, 'GERMANY')),
 ('Aleksandar Pavlovic', (20, 0, 'GERMANY')),
 ('Florian Wirtz', (21, 1, 'GERMANY')),
 ('Jamal Musiala', (21, 2, 'GERMANY')),
 ('Kai Havertz', (24, 16, 'GERMANY'))]

In [19]:
# 5. Sort the results according to the Age
sorted_by_age_rdd = capitalized_rdd.sortBy(lambda x: x[1][0]) # Sort by age (index 0 of the value tuple)

sorted_by_age_rdd.take(5)

[('Aleksandar Pavlovic', (20, 0, 'GERMANY')),
 ('Florian Wirtz', (21, 1, 'GERMANY')),
 ('Jamal Musiala', (21, 2, 'GERMANY')),
 ('Maximilian Beier', (21, 0, 'GERMANY')),
 ('Nico Schlotterbeck', (24, 0, 'GERMANY'))]

In [ ]:
# 6. Average age for country using reduceByKey
# p/ calclar a média = (country, (sum_age, count))
age_sum_count_rdd = capitalized_rdd.map(lambda x:
    (x[1][2], (x[1][0], 1)) # (country, (age, count=1))
)

age_reduced_rdd = age_sum_count_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

average_age_rdd = age_reduced_rdd.map(lambda x:
    (x[0], x[1][0] / x[1][1])
)

average_age_rdd.collect()

[('GERMANY', 21.833333333333332)]

In [ ]:
# 7. Average goals for country reduceByKey
# p/ calcular a média = (country, (sum_goals, count))
goals_sum_count_rdd = capitalized_rdd.map(lambda x:
    (x[1][2], (x[1][1], 1)) # (country, (goals, count=1))
)

goals_reduced_rdd = goals_sum_count_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

average_goals_rdd = goals_reduced_rdd.map(lambda x:
    (x[0], x[1][0] / x[1][1])
)

average_goals_rdd.collect()

[('GERMANY', 3.1666666666666665)]